In [13]:
# pip install sentence-transformers

In [14]:
# !pip install -q \
#     langchain \
#     langchain-core \
#     langchain-community \
#     langchain-google-genai \
#     faiss-cpu>=1.9.0 \
#     sentence-transformers \
#     langchain-classic \
#     langchain-text-splitters \
#     langchain-huggingface

In [15]:
import os 
from pathlib import Path
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document


print("="*60)
print(" INITIATING TWO-STAGE RE-RANKING ARCHITECTURE ")
print("="*60)


 INITIATING TWO-STAGE RE-RANKING ARCHITECTURE 


In [16]:
# Load .env (development) into env vars; production should set real env vars or use a secrets manager.
load_dotenv()  # reads .env if present

# Prefer explicit env var; fall back to a secrets file only if provided
api_key = os.getenv("GEMINI_API_KEY") 

if not api_key:
    secrets_path = Path(os.getenv("SECRETS_PATH", Path("secrets") / "api"))
    if secrets_path.exists():
        with secrets_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("GEMINI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment or secrets file. Set GEMINI_API_KEY or SECRETS_PATH.")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded:", bool(api_key))



API key loaded: True


In [17]:
# Stage 1: The Base Retriever (The Net)
print("[System] Refreshing Vector Database...")
audit_data = [
    Document(page_content="MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.", metadata={"id": 1}),
    Document(page_content="Tuesday Review confirmed the 15% spike is due to 'Legacy Scan-X' hardware and firmware v2.1.", metadata={"id": 2}),
    Document(page_content="Jaymin approved a $45,000 emergency budget to upgrade European scanners by Q1 end.", metadata={"id": 3}),
    Document(page_content="OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.", metadata={"id": 4}),
    Document(page_content="Tony recommends a distributed architecture for handling 500+ PDFs in legacy branches.", metadata={"id": 5}),
    Document(page_content="The 15% error rate is classified as 'Critical' for Banking and Compliance audits.", metadata={"id": 6}),
    Document(page_content="Marten's team is monitoring OCR logs 24/7 until the hardware upgrade is finished.", metadata={"id": 7}),
    Document(page_content="European legacy branches are the only units still using the v2.1 firmware.", metadata={"id": 8}),
    Document(page_content="The new firmware v3.0 has been successfully tested in the North American cluster.", metadata={"id": 9}),
    Document(page_content="Budget allocation for Q1 also includes a 10% reserve for unexpected cloud egress costs.", metadata={"id": 10}),
    Document(page_content="Anisha suggested moving OCR processing to an asynchronous queue using RabbitMQ.", metadata={"id": 11}),
    Document(page_content="Legacy Scan-X machines have a known overheating issue when processing over 100 pages.", metadata={"id": 12}),
    Document(page_content="Compliance team noted that OCR errors are leading to incorrect data in customer KYC files.", metadata={"id": 13}),
    Document(page_content="The upgrade project is codenamed 'Project Vision' and is led by the MLOps Core team.", metadata={"id": 14}),
    Document(page_content="Handwritten PDF recognition accuracy dropped to 62% in the last batch test.", metadata={"id": 15}),
    Document(page_content="Security audit found that legacy firmware v2.1 has three unpatched vulnerabilities.", metadata={"id": 16}),
    Document(page_content="Training data for the new OCR model includes 50,000 samples of handwritten European scripts.", metadata={"id": 17}),
    Document(page_content="The hardware vendor 'OptiScan' has been notified about the hardware failures.", metadata={"id": 18}),
    Document(page_content="A temporary patch was deployed on Monday to reduce memory leaks during batch processing.", metadata={"id": 19}),
    Document(page_content="Q2 Roadmap: Complete migration of all legacy branches to the centralized MLOps platform.", metadata={"id": 20})
]



[System] Refreshing Vector Database...


In [18]:
# Initialize the Brain (Gemini - 2.5 Flash) and Embedder 
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", temperature= 0.0)
embedder = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001")

vectorstore = FAISS.from_documents(audit_data, embedder)


In [19]:
# Stage 1  The Base Retriever (The Net)
base_retriever = vectorstore.as_retriever (search_kwargs = {"k":15})
print("[System] Stage 1 Base Retreiver (FAISS) initialized. Pulling top 15 chunks")

[System] Stage 1 Base Retreiver (FAISS) initialized. Pulling top 15 chunks


In [20]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [21]:
print( "[System] Downloading ms-marco-MiniLM Cross Encoder...")

#1. Inititalize the model 
cross_encoder_model = HuggingFaceCrossEncoder(model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2")

#2. Configure the reranker

reranker = CrossEncoderReranker(model= cross_encoder_model, top_n= 3)

# 3. Build the Two - Stage Retriever 
compression_retriever = ContextualCompressionRetriever(
    base_compressor= reranker,
    base_retriever= base_retriever
)

print("[System] Stage 2 Cross- Encoder initialized.")

[System] Downloading ms-marco-MiniLM Cross Encoder...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7942.47it/s]


[System] Stage 2 Cross- Encoder initialized.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Helper function to format the surviving documents 
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#RAG prompt 
rag_prompt = PromptTemplate.from_template(
    """You are an elite MLOps auditing algorithm. Answer using ONLY this highly-vetted context:

    {context}

    Question: {question}
    Answer:"""
)

print("\n[System] Compling the final LCEL pipeline...")

advanced_rerank_chain = (
    {"context": compression_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("[System] Pipeline ready. Executing test...\n")

user_query = "What is the exact OCR failure rate mentioned in the Q4 audit?"

final_answer = advanced_rerank_chain.invoke(user_query)

print("="*40)
print(f"Query: {user_query}")
print(f" AI Response: {final_answer}")
print("="*40)
print("-> Pipeline Success: 15 chunks retreived, 12 discarded, 3 injected, 0 hallucinations")


[System] Compling the final LCEL pipeline...
[System] Pipeline ready. Executing test...

Query: What is the exact OCR failure rate mentioned in the Q4 audit?
 AI Response: The exact OCR failure rate mentioned in the Q4 audit is 15%.
-> Pipeline Success: 15 chunks retreived, 12 discarded, 3 injected, 0 hallucinations
